# Lab - Unit 7: the speed profile on the real car

**Runs on your machine with CARLA 0.9.12 and `ROAR_PY`.**

On racekit, `agents/unit-07.py` takes 304.15 s on `test-circuit` where Unit 6 took 422.65 s, and it does it at 0 collisions.
It plans every corner with two numbers, A_LAT = 4.6 m/s² and A_DEC = 5.0 m/s², chosen one grid step inside the clean edge of racekit's carla-like car.
**In this lab the real car answers two questions**: does the shipped agent finish three laps of Monza clean, and is there speed left in some sections that the global setting leaves on the table?

You race the shipped agent first, exactly as it is, and record the run.
Then, if it is clean, you tune one speed knob per section with the unit's sequential pass, on a small grid and a fixed run budget.

How to use it:

- The cells tagged `no-exec` are for **your machine** (the Windows PC you set up in the Unit 1 lab).
  Their shell lines are shown with `!` only so the notebook stays valid.
- The analysis cells run anywhere.
  In CI they run against a racekit-made sample recording (the starter on `monza-like`, with no section file), so the book always proves they work; on your machine they run against your own recordings, one at a time.

> **Never commit your results.**
> This repository is public.
> Your recordings (the telemetry CSV **and** its `.sections.txt` sidecar), screenshots and logs stay on your own machine, **outside the repository**
> (the Unit 1 lab uses `%USERPROFILE%\roar-runs\`).
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set `ROAR_LAB_CSV` to one recording's full path (outside the repository) before you start Jupyter; the recorder writes to the same variable, and the section sidecar's name is that path with `.sections.txt` added.

## 1. The five steps

The Unit 1 lab (`units/unit-01-the-race-and-baseline/labs/real-sim.ipynb`) has every detail; this lab only changes what you race and what you record.

1. **Start the Monza v1.1 executable** (CARLA 0.9.12) and wait for the map to load (Unit 1 lab, §1 for the pinned versions and §3 for the start).
2. **Put the submission in the forked competition folder**: copy `agents/unit-07.py` over `ROAR_Competition\competition_code\submission.py`, paste in the Unit 1 recorder with its three wiring lines (Unit 1 lab, §2), make Unit 5's one-line `offset` change (Unit 5 lab, §2), and add the section sidecar of §2 below.
   Before you go to CARLA, race the edited file on racekit and run `check_submission` on it, as the Unit 1 lab says.
3. **Run `competition_runner.py`** from `ROAR_Competition\competition_code`, with `ROAR_LAB_CSV` set to a new file outside the repository.
   Note the time and the collision count the runner reports, and whether the race finished.
4. **Export the telemetry to CSV**: the recorder does it as the car drives, one row per tick, to `ROAR_LAB_CSV`, and writes the sidecar once, when the race starts.
5. **Analyse it**: start Jupyter with `ROAR_LAB_CSV` pointing at the recording and run §3 onwards of this notebook.

§5 says how many runs to make, with which settings, and in which order.

## 2. The recorder: Unit 5's change, and a section sidecar

**Unit 5's change, reused.**
Since Unit 5 the recorder writes the agent's own cross-track error in the `offset` column (`self.cross_track_error`, measured at the start of the tick), and the analysis drops the stale first row after a respawn hold.
`agents/unit-07.py` has the same `initialize()`, the same `advance_index` line and the same `measure()` as `agents/unit-05.py`, so the change and the three wiring lines go in exactly as the Unit 5 lab shows them.

**Why a sidecar.**
This lab reports each of the six **grip sections** separately: per section, how long the car took, how fast it went, how far off the line it got and whether it crashed.
A section is the agent's own idea, `self.section`: a waypoint's distance along the lap from waypoint 0, as a fraction of the lap, times 6, rounded down (the lap cut into six equal pieces of arc length, as racekit cuts it; six is racekit's convention, not a property of Monza).
On CARLA's waypoints, which are not evenly spaced, that is **not** the same as cutting the progress count into six equal slices.
The telemetry CSV's columns are fixed (`racekit.telemetry.COLUMNS`, the v1 format `read_csv` validates), so the section map cannot go in it.
It goes in a second file, written once at `initialize()`: `<the CSV path>.sections.txt`, one integer per waypoint.

**In progress order.**
The recorder's progress does not count from waypoint 0.
It counts from `start_index`, the waypoint nearest the car when the race starts (Unit 1's `start_recording`), so a row's progress p means waypoint (start_index + p) mod n.
The sidecar is therefore written **in progress order**: line k is the section of waypoint (start_index + k) mod n.
Then the analysis reads a row's section as `sections[progress % n]`, which is the right waypoint for any start, on any lap.

Paste the function below after the recorder's constants (it needs nothing but `open`), and add one wiring line to `initialize()`, right after `self.start_recording()`:

```python
    async def initialize(self) -> None:
        location = self.location_sensor.get_last_gym_observation()
        self.current_index = self.nearest_index(location)
        self.start_recording()                                                    # Unit 1
        write_section_sidecar(TELEMETRY_PATH, self.section, self.start_index)    # new
```

`self.section` exists only from Unit 7 on; `self.start_index` is set by `start_recording()`, which is why the new line comes after it.

In [ ]:
# lab-sidecar: the function to paste into submission.py, after the recorder's constants (Python 3.8-safe)
def write_section_sidecar(csv_path, sections, start_index):
    """Write <csv_path>.sections.txt once, at initialize(): one grip section per line, in PROGRESS
    order. Line k is the section of waypoint (start_index + k) mod n, so a telemetry row with
    progress p is in section sections[p % n] on every lap, whatever the start waypoint."""
    n = len(sections)
    with open(f"{csv_path}.sections.txt", "w") as handle:
        for k in range(n):
            handle.write(f"{int(sections[(start_index + k) % n])}\n")

In [ ]:
# Anaconda Prompt, on the Windows machine, from ROAR_Competition\competition_code. Not run in CI.
# Type each line WITHOUT the leading "!" (it is there only so the notebook stays valid).
# 1. The shipped agent (then paste in the recorder, Unit 5's change and the sidecar, as section 2 says)
!copy %PY4KIDS%\roar\projects\racekit\agents\unit-07.py submission.py
# 2. Print the SHA-256 of the file you are about to race, before you edit it, and write it down
!certutil -hashfile %PY4KIDS%\roar\projects\racekit\agents\unit-07.py SHA256
# 3. One run: set ROAR_LAB_CSV and start the runner in ONE command, so the runner's process sees it
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-07-run-1.csv" && python competition_runner.py
# 4. Analyse that run: start Jupyter with the variable pointing at it (from your clone of this repository)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-07-run-1.csv" && jupyter lab

## 3. Analyse one recording, section by section

Set `RACE_FINISHED` to what the runner reported (Unit 1 lab, §4, explains why the recorder cannot know).

Per section the analysis reports:

- **time**: the driving rows (`held == 0`) in that section, times the 0.05 s tick, summed over the laps;
- the **median and peak speed** on those rows;
- the **max |cross-track error|**: the agent's own measurement, on the driving rows without the stale first row after a hold (Unit 5 lab, §3);
- the **collisions**: each hold run counts in the section of the row just before it, where the car crashed.

A row's section is `sections[progress % n]`, with `sections` read from the sidecar.
**Without a sidecar** (an older recorder, or this notebook's CI sample) the analysis falls back to six equal slices of the progress count from the start; it says so, because those slices match the agent's sections only when the waypoints are evenly spaced and the race starts on waypoint 0.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from racekit.telemetry import read_csv

RACE_FINISHED = True  # set False if the runner reported that the race did not finish
CONTROL_DT = 0.05  # s: one row per control tick
SECTIONS = 6
CLEAN_OFFSET = 2.0  # m: the unit's clean rule allows at most this max |cross-track error|


def driving_rows(telemetry, n):
    # The Unit 5 lab's rule: the rows that count for the cross-track statistics, and the stale rows dropped:
    # not held, and not the first row after a hold run (on CARLA it still carries the crash pose).
    # Without held rows, a respawn shows as progress dropping below 0.1 * n; that row is the stale one.
    held = telemetry["held"] == 1
    keep = ~held
    held_rows = np.nonzero(held)[0]
    if len(held_rows):
        stale = [row + 1 for row in held_rows if row + 1 < len(held) and not held[row + 1]]
    else:
        progress = telemetry["progress"]
        stale = [row + 1 for row in np.nonzero((np.diff(progress) < 0) & (progress[1:] < 0.1 * n))[0]]
    keep[np.array(stale, dtype=int)] = False
    return keep, stale


def load_sections(csv_path, n):
    # The recorder's sidecar, <csv_path>.sections.txt: line k is the section of progress k (mod n).
    # Without one, six equal slices of the progress count; returns (sections, True when from the sidecar).
    try:
        with open(f"{csv_path}.sections.txt") as handle:
            sections = [int(handle.readline()) for _ in range(n)]
    except FileNotFoundError:
        return [min(k * SECTIONS // n, SECTIONS - 1) for k in range(n)], False
    return sections, True


def row_sections(progress, sections):
    # The section of every row: sections[progress % n] (progress is cumulative over the laps).
    return np.array(sections)[np.asarray(progress) % len(sections)]


def section_table(telemetry, sections, keep):
    # Per section: time (s), median and peak speed (m/s), max |cross-track error| (m), collisions.
    section = row_sections(telemetry["progress"], sections)
    driving = telemetry["held"] == 0
    held = telemetry["held"] == 1
    starts = [row for row in np.nonzero(held)[0] if row > 0 and not held[row - 1]]
    table = []
    for k in range(SECTIONS):
        rows = driving & (section == k)
        speed = telemetry["v"][rows]
        error = np.abs(telemetry["offset"][rows & keep])
        table.append({
            "time": float(np.sum(rows)) * CONTROL_DT,
            "median_speed": float(np.median(speed)) if len(speed) else float("nan"),
            "peak_speed": float(np.max(speed)) if len(speed) else float("nan"),
            "max_error": float(np.max(error)) if len(error) else float("nan"),
            "collisions": len([row for row in starts if section[row - 1] == k]),
        })
    return table


def print_table(title, table):
    print(title)
    print(f"{'section':>8}{'time (s)':>10}{'median v':>10}{'peak v':>9}{'max |e| (m)':>13}{'collisions':>12}")
    for k, row in enumerate(table):
        print(f"{k:8d}{row['time']:10.2f}{row['median_speed']:10.2f}{row['peak_speed']:9.2f}"
              f"{row['max_error']:13.3f}{row['collisions']:12d}")


telemetry, meta = read_csv(TELEMETRY_PATH)
n = meta["waypoints_per_lap"]
sections, from_sidecar = load_sections(TELEMETRY_PATH, n)
if not from_sidecar:
    print("no .sections.txt sidecar next to this recording: using six equal slices of the progress count,")
    print("which match the agent's sections only on evenly spaced waypoints with the race starting on waypoint 0")
keep, stale = driving_rows(telemetry, n)
recording = section_table(telemetry, sections, keep)
collisions = sum(row["collisions"] for row in recording)
clean = RACE_FINISHED and collisions == 0 and all(row["max_error"] <= CLEAN_OFFSET for row in recording)
print_table(f"your recording: {len(telemetry)} rows, {len(stale)} stale rows dropped", recording)
print(f"clean by the unit's rule (finished, 0 collisions, every section's max |e| <= {CLEAN_OFFSET} m): {clean}")
if max(row["max_error"] for row in recording) == 0.0:
    print("every offset is 0: was this recorded before Unit 5's one-line change?")

In [ ]:
fig, (ax_time, ax_error) = plt.subplots(1, 2, figsize=(12, 3.5))
ax_time.plot(range(SECTIONS), [row["time"] for row in recording], "o-")
ax_time.set_xlabel("grip section")
ax_time.set_ylabel("time in the section (s, all laps)")
ax_time.set_title("where the time goes")
ax_error.plot(range(SECTIONS), [row["max_error"] for row in recording], "o-")
ax_error.axhline(CLEAN_OFFSET, color="grey", linestyle="--", linewidth=1.0)
ax_error.set_xlabel("grip section")
ax_error.set_ylabel("max |cross-track error| (m)")
ax_error.set_title("how close to the clean rule's 2 m (dashed)")
fig.tight_layout()
plt.show()

## 4. racekit's prediction

The shipped agent on racekit's `monza-like` with the carla-like car (`carla_like(noise=True)`, seed 0, `max_seconds=1500`): the reference's seed-0 race of the plan (`roar-006`, U7.4).
This cell analyses it with the **same** functions, and **asserts** its per-section times, peak errors and 0 collisions, so CI proves the table you compare against.

racekit's progress counter starts at the waypoint *after* the start waypoint, so its section map in progress order is the agent's map shifted by one: line k is the section of waypoint k + 1.
`monza-like` is inspired by Monza and is not the CARLA map, and the carla-like car is not CARLA's car: compare the **pattern** (which sections take the time, which run closest to 2 m), not the third decimal.

In [ ]:
from pathlib import Path

from racekit.cars import carla_like
from racekit.runner import run_race
from racekit.tracks import grip_sections, load_track

AGENT = Path("..") / ".." / ".." / "projects" / "racekit" / "agents" / "unit-07.py"
PREDICTED = {  # section: (time in s over the three laps, max |cross-track error| in m), pinned in plan roar-006
    0: (67.85, 0.265), 1: (93.50, 1.624), 2: (102.90, 1.554),
    3: (73.10, 1.672), 4: (92.70, 0.444), 5: (85.05, 1.594),
}
track = load_track("monza-like")
model = run_race(AGENT, track, car=carla_like(noise=True), seed=0, max_seconds=1500.0)
grip = grip_sections(track)
model_sections = [int(grip[(k + 1) % len(grip)]) for k in range(len(grip))]  # racekit's progress order
model_keep, _ = driving_rows(model.telemetry, len(grip))
prediction = section_table(model.telemetry, model_sections, model_keep)
assert model.finished and model.collisions == 0, model
assert abs(model.total_time - 515.10) <= 0.05 + 1e-9, model.total_time
for k, (time_pin, error_pin) in PREDICTED.items():
    assert abs(prediction[k]["time"] - time_pin) <= 0.05 + 1e-9, (k, prediction[k])
    assert abs(prediction[k]["max_error"] - error_pin) <= 0.01 + 1e-9, (k, prediction[k])
    assert prediction[k]["collisions"] == 0, (k, prediction[k])
print_table(f"racekit, monza-like, carla_like(noise=True), seed 0: {model.total_time:.2f} s, clean", prediction)
print()
print(f"{'section':>8}{'time: you':>12}{'racekit':>10}{'max |e|: you':>15}{'racekit':>10}")
for k in range(SECTIONS):
    print(f"{k:8d}{recording[k]['time']:12.2f}{prediction[k]['time']:10.2f}"
          f"{recording[k]['max_error']:15.3f}{prediction[k]['max_error']:10.3f}")

## 5. The decision procedure

The unit's **clean rule**, on CARLA: the runner reports a finished race, 0 collisions, and every section's max |cross-track error| is at most 2 m (§3 prints it).

1. **Race `agents/unit-07.py` exactly as shipped, three laps**, with only the recorder pasted in, and write down its SHA-256 (§2's prompt prints it).
   For the book's author this first run **is** the interim CARLA check of design 013 §7.3, recorded PASS or FAIL in `roar/docs/qualification.md`; a FAIL is surfaced as a follow-up (an errata or fix plan), and no later run replaces it.
2. **Not clean:** lower A_LAT one grid step, to 4.4, and race again; if that is not clean either, to 4.0.
   At most 2 runs.
   This is lab tuning, your path to a usable lab: it never replaces step 1's result.
   If neither is clean, stop here and compare §3 with §4: the sections where your car crashed or ran wide are the first place racekit and CARLA disagree.
3. **Clean:** the **sequential pass** of the lesson, on a coarse grid (1.1, 1.2, 1.3), **one 3-lap run per step**.
   Section 0 first: raise its knob to 1.1, race, and keep going up the grid until a step is not clean or the grid ends; keep the last clean step.
   Then section 1 **on top of** section 0's knob, and so on.
   The **sweep budget is 12 runs**: when it is spent the pass stops, the sections not yet tried stay at 1.0, and the procedure below names them.
   The **short form** sweeps sections 0 and 1 only (at most 6 runs).
4. **Re-race the result twice.**
   If either run is not clean, step **every raised knob back once** (1.3 to 1.2, 1.2 to 1.1, 1.1 to 1.0): the margin rule.
   Then **re-race the stepped-back setting twice**: a setting is accepted only when it has been clean on both of its re-races.
   If one of those is not clean either, stop: the result is all knobs at 1.0 at your A_LAT, marked **provisional**, because no setting was clean on both of its re-races.

The worst case is 1 + 2 + 12 + 4 = 19 runs, about 3 hours at about 10 minutes per run; the short form (at most 1 + 2 + 6 + 4 = 13 runs, and usually far fewer) fits one sitting.
A host failure (the server exits, or the agent gets no first tick) is repeated and not counted: it says nothing about the agent.

**One run per step is weaker evidence than the lesson's five seeds.**
In the lesson a vector that was clean on five seeds failed on two of the next five; a single CARLA run can be lucky in the same way.
That is why step 4 re-races the result, and why the knobs step back if either re-race fails.

**What to look for.**
CARLA's waypoints are not smooth, so their Menger curvature is **spiky**: one waypoint a few centimetres off the line makes a large |κ| where the road hardly bends.
The cap at that waypoint is low, and the backward pass spreads the dip backwards (it can only lower speeds, never raise them), so the profile brakes for a corner that is not there: **phantom braking**.
In §3, a section whose median speed is far below racekit's while its error stays small is a suspect.
Unit 9 replaces the waypoints with a smoothed racing line.

Log every counted run below, in the order you raced them, and run the cell: it replays the procedure on your log, checks that each run is the one the procedure asked for (and that no run is logged after the procedure ended), and prints the next run, the runs used and the sections left untried.

In [ ]:
LAB_GRID = (1.1, 1.2, 1.3)  # the coarse knob grid
FALLBACK_A_LAT = (4.4, 4.0)  # step 2: one grid step down at a time
SWEEP_BUDGET = 12  # step 3: at most this many sweep runs
RE_RACES = 2  # step 4: re-races per setting
STEP_BACKS = 1  # step 4: at most one step back, then its own re-races
SHIPPED_A_LAT = 4.6
SHORT_FORM = False  # True: sweep sections 0 and 1 only

# Your CARLA runs, in the order you raced them: (A_LAT, GRIP, clean by section 3's rule).
RUNS = []


class NextRun(Exception):
    """The procedure needs a run that is not in the log yet."""


def step_back(knob):
    # the margin rule: one grid step down (1.0 stays 1.0)
    lower = [step for step in (1.0,) + LAB_GRID if step < knob]
    return lower[-1] if lower else 1.0


def replay(runs, short_form=False):
    """Replay the procedure on a log of runs; returns (the next run or None, the notes so far)."""
    notes, log = [], list(runs)

    def race(label, a_lat, grip):
        if not log:
            raise NextRun(f"{label}: A_LAT = {a_lat}, GRIP = {tuple(grip)}")
        logged_a_lat, logged_grip, clean = log.pop(0)
        if logged_a_lat != a_lat or tuple(logged_grip) != tuple(grip):
            raise ValueError(f"run {len(runs) - len(log)} is not the run the procedure asked for ({label})")
        notes.append(f"run {len(runs) - len(log):2d}, {label}: {'clean' if clean else 'NOT clean'}")
        return clean

    def finish():
        # the procedure has ended: a run logged after this point is not one it asked for
        if log:
            raise ValueError(f"{len(log)} run(s) logged after the procedure ended; remove them or fix the log")
        return None

    try:
        a_lat, grip = SHIPPED_A_LAT, [1.0] * SECTIONS
        if race("step 1, the shipped agent", a_lat, grip):
            notes.append("step 1: PASS (for the author: the interim check of design 013 section 7.3)")
        else:
            notes.append("step 1: FAIL (for the author: recorded as it is; the fallback below is lab tuning)")
            for lower in FALLBACK_A_LAT:
                a_lat = lower
                if race(f"step 2, A_LAT {lower}", a_lat, grip):
                    break
            else:
                notes.append("step 2: no clean run; stop, and compare sections 3 and 4")
                return finish(), notes
        sweep, untried = 0, []
        for section in ((0, 1) if short_form else range(SECTIONS)):
            if sweep == SWEEP_BUDGET:
                untried.append(section)
                continue
            for knob in LAB_GRID:
                if sweep == SWEEP_BUDGET:
                    notes.append(f"the budget ran out inside section {section}")
                    break
                trial = list(grip)
                trial[section] = knob
                sweep += 1
                if not race(f"step 3, section {section} at {knob}", a_lat, trial):
                    break
                grip = trial
        notes.append(f"step 3: {sweep} sweep runs; the pass keeps GRIP = {tuple(grip)}")
        if untried:
            notes.append(f"the budget was spent: sections {untried} were not tried and stay at 1.0")
        base = [1.0] * SECTIONS
        for round_number in range(1 + STEP_BACKS):
            # all() stops at the first run that is not clean
            if all(race(f"step 4, re-race of GRIP = {tuple(grip)}", a_lat, grip) for _ in range(RE_RACES)):
                notes.append(f"the result: A_LAT = {a_lat}, GRIP = {tuple(grip)}, clean on both re-races")
                break
            if grip == base or round_number == STEP_BACKS:
                notes.append(f"the result is PROVISIONAL: A_LAT = {a_lat}, GRIP = {tuple(base)}; no setting was clean "
                             "on both of its re-races (compare sections 3 and 4)")
                break
            grip = [step_back(knob) for knob in grip]
            notes.append(f"step 4: a re-race was not clean; every raised knob steps back: GRIP = {tuple(grip)}")
        return finish(), notes
    except NextRun as needed:
        return str(needed), notes


# A worked example (made up, not a recording): every run clean, so the budget decides where the pass stops.
example_next, example_notes = replay([(4.6, (1.0,) * 6, True)] + [
    (4.6, tuple(knob if section == s else (1.3 if section < s else 1.0) for section in range(6)), True)
    for s in range(4) for knob in LAB_GRID] + [(4.6, (1.3, 1.3, 1.3, 1.3, 1.0, 1.0), True)] * 2)
assert example_next is None and "sections [4, 5] were not tried" in "\n".join(example_notes)
assert len([note for note in example_notes if note.startswith("run")]) == 15  # 1 + 12 + 2

# Two more checks of the replay: a re-race that fails steps back and re-races the new setting ...
short_runs = [(4.6, (1.0,) * 6, True), (4.6, (1.1,) + (1.0,) * 5, True), (4.6, (1.2,) + (1.0,) * 5, False),
              (4.6, (1.1, 1.1) + (1.0,) * 4, True), (4.6, (1.1, 1.2) + (1.0,) * 4, False),
              (4.6, (1.1, 1.1) + (1.0,) * 4, False)]
stepped_next, _ = replay(short_runs, short_form=True)
assert stepped_next == "step 4, re-race of GRIP = (1.0, 1.0, 1.0, 1.0, 1.0, 1.0): A_LAT = 4.6, GRIP = (1.0, 1.0, 1.0, 1.0, 1.0, 1.0)"
# ... and a run logged after the procedure ended is rejected.
try:
    replay(short_runs + [(4.6, (1.0,) * 6, True)] * 2 + [(4.6, (1.0,) * 6, True)], short_form=True)
except ValueError as extra:
    assert "after the procedure ended" in str(extra)
else:
    raise AssertionError("an extra run was accepted")

next_run, notes = replay(RUNS, SHORT_FORM)
print("\n".join(notes) if notes else "no runs logged yet")
max_sweep = min(SWEEP_BUDGET, 2 * len(LAB_GRID)) if SHORT_FORM else SWEEP_BUDGET
print(f"runs used: {len(RUNS)} of at most {1 + len(FALLBACK_A_LAT) + max_sweep + RE_RACES * (1 + STEP_BACKS)}")
print(f"next: {next_run}" if next_run else "the procedure is complete")

## What to take away

- The first run of the shipped agent is a **measurement**, recorded as it is; everything after it is tuning, and tuning never rewrites it.
- The per-section table tells you *where* the real car differs from racekit's: compare its pattern with §4, section by section.
- A knob is a speed knob, not a grip estimate: a knob of 1.3 means the planner was too cautious in that section on *this* car with *this* line, for whatever reason.
  Unit 8 measures μ from the logs.
- Write down every run, clean or not, with its settings: a tuning made on one run per step and never re-raced is a guess.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, their `.sections.txt` sidecars, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at.
> Run `git status` in your clone before you commit anything: none of them should be listed.